# WulfPak Seed Data Generator

Generates realistic fake contacts using the Gemini API, output as a WulfPak backup JSON file.

**To run on Kaggle:**
1. Add your Gemini API key under *Add-ons → Secrets* with the name `GEMINI_API_KEY`
2. Run All → download the output `.json` from the output panel
3. On your phone: *Settings → Contacts → Import backup*

**To run locally:**
```bash
uv run --with google-generativeai jupyter nbconvert --to script generate_seed_data.ipynb --execute
# or just run the .py version:
uv run --with google-generativeai python generate_seed_data.py
```

In [ ]:
!pip install -q google-generativeai

In [ ]:
import json
import uuid
import os
import random
from datetime import datetime, timezone, timedelta
import google.generativeai as genai

# ── API key ───────────────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
    print("Loaded API key from Kaggle secrets")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
    print("Loaded API key from environment variable")

if not GEMINI_API_KEY:
    raise RuntimeError("Set GEMINI_API_KEY as a Kaggle secret or environment variable")

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.0-flash")
print("Gemini client ready")

In [ ]:
# ── Constants and helpers ─────────────────────────────────────────────────────

DB_VERSION = 8
NOW_MS = int(datetime.now(timezone.utc).timestamp() * 1000)

REL_LABELS = [
    "friend", "best_friend", "acquaintance", "romantic_partner",
    "mother", "father", "sibling", "child", "grandparent", "cousin", "aunt", "uncle",
    "colleague", "manager", "report", "mentor", "client",
]

FAMILY_LABELS = {"mother", "father", "sibling", "child", "grandparent", "cousin", "aunt", "uncle"}
SOCIAL_LABELS = {"friend", "best_friend", "acquaintance", "romantic_partner"}
WORK_LABELS   = {"colleague", "manager", "report", "mentor", "client"}

def rel_category(label):
    if label in FAMILY_LABELS: return "FAMILY"
    if label in SOCIAL_LABELS: return "SOCIAL"
    if label in WORK_LABELS:   return "WORK"
    return "OTHER"

def make_uuid():
    return str(uuid.uuid4())

def ms_days_ago(days):
    return int((datetime.now(timezone.utc) - timedelta(days=days)).timestamp() * 1000)

def gemini_json(prompt):
    resp = model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(response_mime_type="application/json"),
    )
    return json.loads(resp.text)

print("Helpers defined")

## Step 1 — "Me" person

In [ ]:
me = {
    "id": make_uuid(),
    "firstName": "Demo",
    "lastName": "User",
    "nickname": None,
    "photoUri": None,
    "relationLabel": "friend",
    "isFavorite": False,
    "lastContactedAt": None,
    "interactionCount": 0,
    "closenessRating": None,
    "company": None,
    "jobTitle": None,
    "cachedSummary": None,
    "summaryGeneratedAt": None,
    "closenessScore": None,
    "isMe": True,
}
print(f"Me person id: {me['id']}")

## Step 2 — Generate 22 contacts

In [ ]:
PERSONS_PROMPT = """
Generate a JSON array of 22 realistic fake people for a personal CRM app.
Use culturally diverse names (mix of American, European, Asian, Latino backgrounds).
No real people — completely fictional.

Each person object must have exactly these fields:
- firstName (string)
- lastName (string or null)
- nickname (string or null, ~25% have one)
- relationLabel: one of: {labels}
- isFavorite (boolean, ~20% true)
- company (string or null)
- jobTitle (string or null)
- closenessRating (integer 1-5 or null, ~60% have one)

Desired mix:
- 2 family: one mother or father, one sibling or cousin
- 1 romantic_partner or best_friend
- 5 friends
- 3 acquaintances
- 6 colleagues (mix of colleague, manager, report)
- 2 work-adjacent (mentor or client)
- 3 miscellaneous

Return ONLY a valid JSON array — no markdown, no explanation.
""".format(labels=", ".join(REL_LABELS))

raw_persons = gemini_json(PERSONS_PROMPT)
persons = []
for p in raw_persons:
    persons.append({
        "id": make_uuid(),
        "firstName": p["firstName"],
        "lastName": p.get("lastName"),
        "nickname": p.get("nickname"),
        "photoUri": None,
        "relationLabel": p["relationLabel"],
        "isFavorite": p.get("isFavorite", False),
        "lastContactedAt": None,
        "interactionCount": 0,
        "closenessRating": p.get("closenessRating"),
        "company": p.get("company"),
        "jobTitle": p.get("jobTitle"),
        "cachedSummary": None,
        "summaryGeneratedAt": None,
        "closenessScore": None,
        "isMe": False,
    })

all_persons = [me] + persons
non_me = persons
print(f"Generated {len(persons)} contacts")
for p in persons[:5]:
    print(f"  {p['firstName']} {p['lastName'] or ''} — {p['relationLabel']}")
print("  ...")

## Step 3 — Contact details

In [ ]:
CONTACT_DETAILS_PROMPT = """
Generate contact details for these people. Use the relation to decide what's realistic:
- family → phone (mobile) + maybe email
- friends → phone (mobile), ~50% also have email
- colleagues → email (work) + maybe phone, ~30% have LinkedIn social
- acquaintances → just phone or just email
- mentor/client → email + maybe LinkedIn

Each contact detail:
- personId (from input)
- type: PHONE, EMAIL, SOCIAL, or ADDRESS
- label: e.g. "mobile", "home", "work email", "instagram", "linkedin"
- value: realistic fake value (fake phones like (555) 867-5309, fake emails like name@domain.com)

People: {summary}

Return ONLY a valid JSON array of objects with fields: personId, type, label, value.
"""

summary = [{"id": p["id"], "firstName": p["firstName"], "lastName": p["lastName"], "relationLabel": p["relationLabel"]} for p in non_me]
raw_details = gemini_json(CONTACT_DETAILS_PROMPT.format(summary=json.dumps(summary)))
contact_details = [{"id": make_uuid(), "personId": d["personId"], "type": d["type"], "label": d["label"], "value": d["value"]} for d in raw_details]
print(f"Generated {len(contact_details)} contact details")

## Step 4 — Life events

In [ ]:
LIFE_EVENTS_PROMPT = """
Generate life events for these people.

Rules:
- Everyone gets a birthday (use realistic birth years 1950-2002)
- romantic_partner gets an anniversary
- ~40% of friends/family also get a graduation, job_change, or moved event

eventType values: birthday, anniversary, graduation, job_change, moved

People: {summary}

Each event:
- personId
- eventType
- date: Unix timestamp in milliseconds (realistic past date)
- isRecurring: true for birthday/anniversary, false otherwise
- note: optional short string or null

Return ONLY a valid JSON array.
"""

summary = [{"id": p["id"], "firstName": p["firstName"], "relationLabel": p["relationLabel"]} for p in non_me]
raw_events = gemini_json(LIFE_EVENTS_PROMPT.format(summary=json.dumps(summary)))
life_events = [{"id": make_uuid(), "personId": e["personId"], "eventType": e["eventType"], "date": e["date"], "isRecurring": e.get("isRecurring", False), "note": e.get("note")} for e in raw_events]
print(f"Generated {len(life_events)} life events")

## Step 5 — Interactions

In [ ]:
INTERACTIONS_PROMPT = """
Generate interaction history for these contacts in a personal CRM.

Interaction types: call, text, email, video_call, in_person, social_media

Frequency guidelines:
- best_friend / family: 8-12 interactions over the past year
- friend: 4-7 interactions
- colleague / manager: 3-5 interactions, mostly email/call
- acquaintance: 1-2 interactions

For each interaction:
- personId (contact it was with)
- type (one of the types above)
- daysAgo: integer 1-400
- durationSeconds: integer or null (for call/video_call: 120-3600, else null)
- note: short realistic note string or null (~35% should have notes)

People: {summary}

Return ONLY a valid JSON array.
"""

summary = [{"id": p["id"], "firstName": p["firstName"], "relationLabel": p["relationLabel"]} for p in non_me]
raw_interactions = gemini_json(INTERACTIONS_PROMPT.format(summary=json.dumps(summary)))

interactions = []
i_participants = []
for item in raw_interactions:
    iid = make_uuid()
    interactions.append({
        "id": iid,
        "timestamp": ms_days_ago(item.get("daysAgo", random.randint(1, 200))),
        "type": item["type"],
        "durationSeconds": item.get("durationSeconds"),
        "note": item.get("note"),
    })
    i_participants.append({"interactionId": iid, "personId": item["personId"]})

print(f"Generated {len(interactions)} interactions")

## Step 6 — Notes

In [ ]:
NOTES_PROMPT = """
Generate personal notes about these contacts in a CRM. Notes capture things worth
remembering: preferences, allergies, mentioned projects, names of their kids, etc.

Quantity:
- best_friend / family: 3-5 notes
- friend: 2-3 notes
- colleague: 1-2 notes
- acquaintance: 0-1 notes

Each note:
- personId
- daysAgo: integer 1-600
- body: 1-3 sentence note text

People: {summary}

Return ONLY a valid JSON array.
"""

summary = [{"id": p["id"], "firstName": p["firstName"], "relationLabel": p["relationLabel"]} for p in non_me]
raw_notes = gemini_json(NOTES_PROMPT.format(summary=json.dumps(summary)))
notes = [{"id": make_uuid(), "personId": n["personId"], "timestamp": ms_days_ago(n.get("daysAgo", 60)), "body": n["body"]} for n in raw_notes]
print(f"Generated {len(notes)} notes")

## Step 7 — Gifts

In [ ]:
GIFTS_PROMPT = """
Generate gift ideas and gift history for close contacts.

Focus on:
- best_friend / romantic_partner: 2-3 gifts
- family / friend: 1-2 gifts
- colleague: 0-1 gifts (office-appropriate)

Each gift:
- personId
- name: gift name
- occasion: birthday, christmas, anniversary, graduation, or null
- status: IDEA, PURCHASED, or GIVEN
- note: optional note or null

People: {summary}

Return ONLY a valid JSON array.
"""

close = [p for p in non_me if p["relationLabel"] in {"best_friend", "romantic_partner", "friend", "mother", "father", "sibling"}]
summary = [{"id": p["id"], "firstName": p["firstName"], "relationLabel": p["relationLabel"]} for p in close]
raw_gifts = gemini_json(GIFTS_PROMPT.format(summary=json.dumps(summary)))
gifts = [{"id": make_uuid(), "personId": g["personId"], "name": g["name"], "occasion": g.get("occasion"), "status": g.get("status", "IDEA"), "note": g.get("note")} for g in raw_gifts]
print(f"Generated {len(gifts)} gifts")

## Step 8 — Shared activities

In [ ]:
ACTIVITIES_PROMPT = """
Generate 6-8 shared activities/events that involve multiple people from this list.
Examples: "Game night at Jake's", "Team offsite in Austin", "Hiking trip", "Birthday dinner for Maria".

Each activity:
- title: short event title
- body: 1-2 sentence description or null
- daysAgo: integer 1-365
- participantIds: list of 2-4 person IDs from the input who attended

People available: {summary}

Return ONLY a valid JSON array with fields: title, body, daysAgo, participantIds.
"""

summary = [{"id": p["id"], "firstName": p["firstName"], "relationLabel": p["relationLabel"]} for p in non_me]
raw_activities = gemini_json(ACTIVITIES_PROMPT.format(summary=json.dumps(summary)))

activities = []
a_participants = []
for item in raw_activities:
    aid = make_uuid()
    activities.append({
        "id": aid,
        "timestamp": ms_days_ago(item.get("daysAgo", random.randint(7, 180))),
        "title": item["title"],
        "body": item.get("body"),
    })
    for pid in item.get("participantIds", []):
        a_participants.append({"activityId": aid, "personId": pid})

print(f"Generated {len(activities)} activities")

## Step 9 — Inter-contact relationships

In [ ]:
RELATIONSHIPS_PROMPT = """
Given these contacts, generate 5-8 inter-contact relationships (people who know each other).
For example: two siblings both in the list, coworkers at the same company, mutual friends.

Important: personAId MUST be lexicographically less than personBId (string sort on UUIDs).
Label is from personA's perspective.

Use only these labels: friend, best_friend, acquaintance, sibling, romantic_partner, colleague, cousin

People: {summary}

Return ONLY a valid JSON array with fields: personAId, personBId, label.
"""

summary = [{"id": p["id"], "firstName": p["firstName"], "relationLabel": p["relationLabel"]} for p in non_me]
raw_rels = gemini_json(RELATIONSHIPS_PROMPT.format(summary=json.dumps(summary)))

person_relationships = []
seen = set()
for r in raw_rels:
    a, b = r["personAId"], r["personBId"]
    if a > b:
        a, b = b, a
    key = (a, b)
    if key in seen:
        continue
    seen.add(key)
    label = r["label"]
    person_relationships.append({"personAId": a, "personBId": b, "label": label, "category": rel_category(label), "relType": None})

print(f"Generated {len(person_relationships)} inter-contact relationships")

## Step 10 — Assemble and write backup file

In [ ]:
# Update interactionCount and lastContactedAt on each person
from collections import defaultdict
counts    = defaultdict(int)
last_ts   = defaultdict(int)
iid_to_ts = {i["id"]: i["timestamp"] for i in interactions}

for ip in i_participants:
    pid = ip["personId"]
    ts  = iid_to_ts.get(ip["interactionId"], 0)
    counts[pid] += 1
    if ts > last_ts[pid]:
        last_ts[pid] = ts

for p in all_persons:
    if p["id"] in counts:
        p["interactionCount"]  = counts[p["id"]]
        p["lastContactedAt"]   = last_ts[p["id"]]

backup = {
    "version": DB_VERSION,
    "exportedAt": NOW_MS,
    "persons": all_persons,
    "personRelationships": person_relationships,
    "contactDetails": contact_details,
    "interactions": interactions,
    "interactionParticipants": i_participants,
    "notes": notes,
    "lifeEvents": life_events,
    "activities": activities,
    "activityParticipants": a_participants,
    "gifts": gifts,
    "tasks": [],
}

ts_str   = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"wulfpak_seed_{ts_str}.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(backup, f, indent=2, ensure_ascii=False)

print(f"\n✓ Output: {filename}")
print(f"  Persons        : {len(all_persons)} (including you)")
print(f"  Interactions   : {len(interactions)}")
print(f"  Notes          : {len(notes)}")
print(f"  Life events    : {len(life_events)}")
print(f"  Contact details: {len(contact_details)}")
print(f"  Gifts          : {len(gifts)}")
print(f"  Activities     : {len(activities)}")
print(f"  Relationships  : {len(person_relationships)}")
print(f"\nImport: Settings → Contacts → Import backup")